# ESI HV module activation probe

This notebook isolates the three independent HV conditions exposed by `COM-ESI-CTRL.dll`: target voltage, controller-wide `SetEnable`, and module-level `SetModuleActivationState`.

## Safety

- Run on the Windows PC connected to the ESI controller.
- Close ESIBD Explorer and ESI-Control first; the DLL is single-instance.
- Keep the hardware interlock available and use an external meter.
- The default configuration never writes a nonzero target. It first writes `0 V` to both HV modules, then disables the global gate before toggling module state.
- The probe does not restore previous output states. Its finalizer writes `0 V`, requests module standby, disables the global gate, and closes the COM port.
- A nonzero test requires changing `ARM_NONZERO_TEST` to `True`. Start at `10 V`.
- The vendor DLL calls are not cancellable. Restart the Jupyter kernel if a call blocks.


In [ ]:
# Adjust these values only while ESIBD Explorer is closed.
COM_PORT = 16
BAUD_RATE = 230400
MODULE_ADDRESS = 1  # HV1=1, HV2=2
ARM_NONZERO_TEST = False
TEST_VOLTAGE = 10.0
SETTLE_SECONDS = 2.0

from pathlib import Path

candidates = [
    Path.cwd(),
    Path.cwd() / 'esi',
    Path(r'C:/Users/lab_admin/ESIBD Explorer/plugins/esi'),
]
PLUGIN_DIR = next(
    (path for path in candidates if (path / 'esi_plugin.py').is_file()),
    None,
)
if PLUGIN_DIR is None:
    raise FileNotFoundError('Could not locate the installed esi plugin folder.')

DRIVER_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'esi' / 'esi_base.py'
DLL_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'esi' / 'vendor' / 'x64' / 'COM-ESI-CTRL.dll'
ERROR_CODES_FILE = PLUGIN_DIR / 'vendor' / 'runtime' / 'error_codes.json'
print(f'Plugin: {PLUGIN_DIR}')
print(f'COM{COM_PORT} @ {BAUD_RATE} baud; module={MODULE_ADDRESS}')
print(f'Nonzero test armed: {ARM_NONZERO_TEST}')


In [ ]:
import importlib.util
import json
import time
from datetime import datetime

spec = importlib.util.spec_from_file_location('_esi_activation_probe_base', DRIVER_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f'Could not load {DRIVER_FILE}')
driver_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(driver_module)
ESIBase = driver_module.ESIBase

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_safe(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

def record_call(device, action, method, *args, required=False):
    try:
        result = method(*args)
        if isinstance(result, tuple):
            status, values = int(result[0]), list(result[1:])
        else:
            status, values = int(result), []
        record = {
            'status': status,
            'status_text': device.format_status(status),
            'values': json_safe(values),
        }
    except Exception as exc:
        record = {'error': f'{type(exc).__name__}: {exc}'}
        if required:
            raise
        return record
    if required and status != device.NO_ERR:
        raise RuntimeError(f'{action} failed: {device.format_status(status)}')
    return record

def read_hv_state(device, address):
    state = {
        'global_enable': record_call(device, 'get_enable', device.get_enable),
        'main_state': record_call(device, 'get_main_state', device.get_main_state),
        'interlock_state': record_call(device, 'get_interlock_state', device.get_interlock_state),
        'interlock_enable': record_call(device, 'get_interlock_enable', device.get_interlock_enable),
        'inputs': record_call(device, 'get_inputs', device.get_inputs),
        'module_state': record_call(device, 'get_module_state', device.get_module_state, address),
        'module_activation_direct': record_call(
            device, 'get_module_activation_state',
            device.get_module_activation_state, address
        ),
        'module_led': record_call(device, 'get_module_led_data', device.get_module_led_data, address),
        'pwm': record_call(device, 'get_hv_supply_params_pwm', device.get_hv_supply_params_pwm, address),
        'target': record_call(
            device, 'get_hv_supply_target_output_voltage',
            device.get_hv_supply_target_output_voltage, address
        ),
        'measured_voltage': record_call(
            device, 'get_hv_supply_output_voltage',
            device.get_hv_supply_output_voltage, address
        ),
        'measured_current': record_call(
            device, 'get_hv_supply_output_current',
            device.get_hv_supply_output_current, address
        ),
    }
    pwm_values = state['pwm'].get('values', [])
    state['pwm_activation'] = bool(pwm_values[6]) if len(pwm_values) >= 7 else None
    target_values = state['target'].get('values', [])
    state['target_v'] = float(target_values[0]) if target_values else None
    enable_values = state['global_enable'].get('values', [])
    state['global_enabled'] = bool(enable_values[0]) if enable_values else None
    print(
        f"global={state['global_enabled']}, "
        f"module_pwm_active={state['pwm_activation']}, "
        f"target={state['target_v']} V, "
        f"measured={state['measured_voltage'].get('values')}"
    )
    return state

print('Driver loaded. No COM port has been opened yet.')


## Run the activation probe

At `0 V`, observe whether the module LED changes between the requested STANDBY and ACTIVE states. The direct activation getter may return `-10` on firmware `0x0100`; the decisive independent value is `pwm_activation` from `GetHVsupplyParamsPWM`.


In [ ]:
def run_activation_probe():
    if MODULE_ADDRESS not in (1, 2):
        raise ValueError('MODULE_ADDRESS must be 1 or 2.')
    if not 0.0 <= float(TEST_VOLTAGE) <= 10.0:
        raise ValueError('Keep TEST_VOLTAGE between 0 and 10 V for this probe.')

    device = ESIBase(
        com=COM_PORT,
        idn='esi_hv_activation_probe',
        dll_path=DLL_FILE,
        error_codes_path=ERROR_CODES_FILE,
    )
    opened = False
    report = {
        'timestamp': datetime.now().astimezone().isoformat(),
        'com_port': COM_PORT,
        'module_address': MODULE_ADDRESS,
        'armed_nonzero': bool(ARM_NONZERO_TEST),
        'test_voltage_v': float(TEST_VOLTAGE),
        'steps': {},
        'cleanup': {},
    }
    try:
        report['steps']['open'] = record_call(
            device, 'open_port', device.open_port, COM_PORT, required=True
        )
        opened = True
        report['steps']['baud'] = record_call(
            device, 'set_comspeed', device.set_comspeed, BAUD_RATE, required=True
        )

        for address in (1, 2):
            report['steps'][f'zero_target_{address}'] = record_call(
                device, f'zero_target_{address}',
                device.set_hv_supply_target_output_voltage, address, 0.0,
                required=True,
            )
        report['steps']['zero_heat'] = record_call(
            device, 'zero_heat', device.set_heat_ctrl_heater_temperature, 0.0
        )
        report['steps']['global_off_initial'] = record_call(
            device, 'set_enable(False)', device.set_enable, False, required=True
        )
        report['steps']['global_on_for_test'] = record_call(
            device, 'set_enable(True)', device.set_enable, True, required=True
        )
        report['steps']['baseline'] = read_hv_state(device, MODULE_ADDRESS)

        report['steps']['request_standby'] = record_call(
            device, 'set_module_activation_state(False)',
            device.set_module_activation_state, MODULE_ADDRESS, False
        )
        time.sleep(SETTLE_SECONDS)
        report['steps']['standby'] = read_hv_state(device, MODULE_ADDRESS)
        input('Observe the module LED in STANDBY, then press Enter...')

        report['steps']['request_active'] = record_call(
            device, 'set_module_activation_state(True)',
            device.set_module_activation_state, MODULE_ADDRESS, True
        )
        time.sleep(SETTLE_SECONDS)
        report['steps']['active_zero_v'] = read_hv_state(device, MODULE_ADDRESS)
        input('Observe the module LED in ACTIVE at 0 V, then press Enter...')

        active_state = report['steps']['active_zero_v']
        if ARM_NONZERO_TEST:
            if active_state['pwm_activation'] is not True:
                raise RuntimeError('PWM status did not confirm module activation.')
            if active_state['global_enabled'] is not True:
                raise RuntimeError('Global enable readback is not ON.')
            report['steps']['set_test_voltage'] = record_call(
                device, 'set_test_voltage',
                device.set_hv_supply_target_output_voltage,
                MODULE_ADDRESS, float(TEST_VOLTAGE), required=True,
            )
            time.sleep(SETTLE_SECONDS)
            report['steps']['nonzero'] = read_hv_state(device, MODULE_ADDRESS)
            input('Record both physical output voltages, then press Enter to force OFF...')

        return report
    finally:
        if opened:
            for address in (1, 2):
                report['cleanup'][f'zero_target_{address}'] = record_call(
                    device, f'cleanup_zero_target_{address}',
                    device.set_hv_supply_target_output_voltage, address, 0.0
                )
                report['cleanup'][f'standby_{address}'] = record_call(
                    device, f'cleanup_standby_{address}',
                    device.set_module_activation_state, address, False
                )
            report['cleanup']['global_off'] = record_call(
                device, 'cleanup_global_off', device.set_enable, False
            )
            report['cleanup']['close'] = record_call(
                device, 'close_port', device.close_port
            )
        report_path = PLUGIN_DIR / f"esi_hv_activation_probe_{datetime.now():%Y%m%d_%H%M%S}.json"
        report_path.write_text(
            json.dumps(json_safe(report), indent=2), encoding='utf-8'
        )
        print(f'Report written to {report_path}')

activation_report = run_activation_probe()


## Interpretation

- `request_active.status == -10` with `active_zero_v.pwm_activation == true`: firmware omitted the direct acknowledgement, but the HVC toggle was applied.
- `pwm_activation` remains false: the module activation command was not applied; do not arm the nonzero test.
- `pwm_activation == true`, `global_enabled == true`, and target readback is correct, but physical voltage remains zero: inspect controller state, interlocks, input levels, and module LED values in the JSON report.
- Compare the LED observation between `standby` and `active_zero_v`; yellow in both states indicates that module activation is still inhibited.
